# Minimum Current Estimate (MCE) methods

In [1]:
import mindmap as mm

dm = mm.io.TutoManager("../../config.yml")
sub = "simf10"

# data
fwd = dm.load(sub, "fwd", desc="inverse")
ave = dm.load(sub, "ave", task="rest", run="11")
ncov = dm.load(sub, "cov", task="rest", run="11")
gold = dm.load(sub, "gold", task="rest", run="11")

In [ ]:
import numpy as np
import pyxu.operator as pxo
import pyxu.opt.solver as pxs
import pyxu.opt.stop as pxst
from pyxu.abc import LinOp

from mindmap.analysis.functional.inverse import MNE, make_stc


class MCE(MNE):
    """Minimum Current Estimate (L1-norm) sovler using Pyxu."""
    def __init__(self, forward, noise_cov, depth=0.8, verbose="warning"):
        # Inherit MNE to reuse foward solution conversion and gain extraction
        super().__init__(forward, noise_cov, depth=depth, noise_norm=None, verbose=verbose)

    def solve(self, evoked, alpha=0.1, max_iter=500, rel_grad_tol=1e-4):
        """Solve L1 inverse problem: min ||y-Gx||^2_2 + alpha * ||Wx||_1"""
        # 1. Preprocessing (Reuse MNE logic)
        projs_gain, projs_data, whitener = self._setup_preprocessor(evoked.info, evoked.nave)

        # 2. Whiten Gain and Data
        gain_white = whitener @ projs_gain @ self.gain_raw
        y_white = whitener @ projs_data @ evoked.data

        # 3. Compute Source Prior (Depth Weighting)
        # In MCE, the prior W is usually applied as a diagonal scaling
        w_source = self.compute_source_prior(gain_white)

        # 4. Construct Pyxu Operators
        # We solve for x_weighted, where x = w_source * x_weighted
        if w_source.ndim == 1:
            G = LinOp.from_array(gain_white * w_source)
        else:
            G = LinOp.from_array(gain_white @ w_source)


        # Regularization: alpha * ||x||_1
        penalty = alpha * pxo.L1Norm(dim_shape=G.dim_shape)

        # solution storage
        sol = np.zeros((G.dim_shape[0], y_white.shape[1]))

        # stopping criterion
        stop_crit = pxst.RelError(eps=rel_grad_tol, var="x") | pxst.MaxIter(max_iter)

        # Iterate over time points
        for i, y in enumerate(y_white.T):
            # Loss: ||y-Gx||^2_2
            loss = pxo.SquaredL2Norm(dim_shape=G.codim_shape).argshift(-y) * G
            loss.diff_lipschitz = loss.estimate_diff_lipschitz()

            # 5. Optimization (APGD / FISTA)
            # MCE is typically solved for each time point
            solver = pxs.PGD(loss, penalty, show_progress=False)
            solver.fit(
                x0 = np.zeros(G.dim_shape),
                stop_crit = stop_crit,
            )

            sol_weighted = solver.solution().squeeze()

            # 6. Map back to original space
            if w_source.ndim == 1:
                sol[:,i] = sol_weighted * w_source
            else:
                sol[:,i] = w_source @ sol_weighted

        return make_stc(
            source_space = self.forward["src"],
            activity = sol,
            tmin = evoked.times[0],
            tstep = 1/evoked.info["sfreq"],
        )
    
class CSET(MCE):
    """Connectome Spectrum Electromagnetic Tomography"""
    def __init__(self, forward, eigvec, noise_cov, depth=0.8, verbose="warning"):
        """init"""
        super().__init__(forward, noise_cov, depth, verbose="warning")
        self.eigvec = eigvec

    def compute_source_prior(self):
        """"""
        return

In [6]:
inv = MCE(fwd, ncov)
sol = inv.solve(ave, alpha=0.1, max_iter=5)

In [ ]:
mm.viz.plot_brain_activity(fwd["src"], [sol, gold], func_params={"absolute":True, "norm":"minmax"}).show()

array([-4.00001383e-03, -1.38282776e-08,  3.99998617e-03])

In [8]:
gold.times

array([0.0000e+00, 4.0000e-03, 8.0000e-03, ..., 1.9988e+01, 1.9992e+01,
       1.9996e+01])